# DIFF — is the leftover-task computed *at the pivot* by attention over TASK, keyed by PROGRESS?

At the end of a code stub the model must encode `DIFF = TASK − PROGRESS`. If a goal/decision head does
this *at its own layer* via attention, the three projections split the labor:

| projection | at | encodes | governs |
|---|---|---|---|
| **QUERY** | pivot (last stub token) | **PROGRESS** | *where to look* — the selection |
| **KEY**   | task-description tokens  | TASK (addressing) | *attention pattern* |
| **VALUE** | task-description tokens  | TASK (content)    | *what identity is written* |

Two contrastive transplants, in **separate sections** below (different analysis strategies):

- **SPEC 1 — same TASK, different PROGRESS → transplant QUERY @ pivot.** Keys/values (task) are shared
  (identical tokens *and positions* — the task precedes the differing prefill), so only the pivot query
  differs. Per-q-head, single position, **no GQA bleed**. Test A compares patched vs **clean DEST** over
  the shared task.
- **SPEC 2 — different TASK, same PROGRESS → transplant KEY + VALUE @ task span.** KEY re-addresses,
  **VALUE re-fills** the content (KEY alone leaves DEST values → DEST identity). Per-kv-head (GQA bleed).
  Test A compares patched vs **clean SRC** over the SOURCE task tokens (clipped).

**Uniform prediction (both):** the transplanted DEST pivot readout ≈ **clean SRC**.

Caveats: q/k patch is pre-RoPE (dest RoPE supplies position → assumes ~aligned spans); k/v carry GQA
bleed (queries exempt); default patch layers = the selected heads' own layers.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, logging
sys.path.append("../..")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "4, 7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import numpy as np
from collections import defaultdict
from contextlib import contextmanager, nullcontext
from rich.console import Console, Group
from rich.theme import Theme
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich.box import SIMPLE_HEAVY
from rich.markup import escape
from src.utils import logging_utils, env_utils
from src.goals.head_loc_attn import HeadMask

logging_utils.configure(level=logging.WARNING)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)
RES = env_utils.DEFAULT_RESULTS_DIR
GQA_BLEED     = False                      # False = correct the GQA bleed (per-query-head routing:
                                           #   only PATCH_HEADS_2 read the source span, others clean);
                                           # True  = raw per-kv-head patch (bleed present) — original.
                                           # Affects GENERATION (2C) only; 2A/2B read just the selected
                                           # heads, so they are bleed-immune either way.

In [2]:
# ── model ──
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(mt.name)
N_KV = mt.model.config.num_key_value_heads
GROUP = mt.n_heads // N_KV
print(f"n_heads={mt.n_heads}  n_kv_heads={N_KV}  group={GROUP}  head_dim={mt.head_dim}  n_layer={mt.n_layer}")

2026-09-05 16:53:06 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64


In [3]:
# ── heads: single source of truth ───────────────────────────────────────────
# GOAL heads (READOUT) — trained localization mask (lam20), mid-band, thresholded.
# BOUNDARY heads (DECISION) — move-on/stay separation (cos < thr) from clock/15's aff_sep.
import json

N_LAYER = mt.n_layer
N_HEADS = mt.model.config.num_attention_heads

def mask_bin(heads):
    m = HeadMask(N_LAYER, N_HEADS)
    with torch.no_grad():
        m.values.zero_()
        for l, h in heads:
            m.values[l, h] = 1.0
    return m

GOAL_THRESHOLD = 0.9
_gckpt = torch.load(os.path.join(RES, f"localization/{MODEL_NAME.split('/')[-1]}/lam20/head_mask.pt"), map_location="cpu")
GOAL_LAYERS = _gckpt["layers"]
_goal_trained = HeadMask(len(GOAL_LAYERS), _gckpt["mask_values"].shape[1])
with torch.no_grad():
    _goal_trained.values.copy_(_gckpt["mask_values"])
    if MODEL_NAME.split("/")[-1] == "Qwen2.5-Coder-32B-Instruct":
        _goal_trained.values[0:25] = 0
        _goal_trained.values[56:] = 0
GOAL_HEADS = sorted(_goal_trained.selected_heads(GOAL_LAYERS, threshold=GOAL_THRESHOLD))
GOAL_MASK  = mask_bin(GOAL_HEADS)

from src.goals.oid_gaze import cache_root
BOUNDARY_SEL_THR = 0.8
_bnd_root = os.path.join(cache_root(MODEL_NAME, "conditions"), "completion_mask")
_bnd_dir = max((t for t in os.listdir(_bnd_root)
                if os.path.isdir(os.path.join(_bnd_root, t)) and "lam10.0" in t),
               key=lambda t: os.path.getmtime(os.path.join(_bnd_root, t)))
_DCK = torch.load(os.path.join(_bnd_root, _bnd_dir, "mask.pt"), map_location="cpu", weights_only=False)
_dec_layers = [int(x) for x in _DCK["layers"]]
_dmv = _DCK["mask_values"].numpy()
BOUNDARY_ALL = sorted((_dec_layers[li], int(h)) for li, h in np.argwhere(_dmv > BOUNDARY_SEL_THR))
BOUNDARY_COS_THR = 0.7
_aff = torch.load(os.path.join(RES, "notebook_caches/15_order_dependency/aff_sep__d5fbad08ba.pt"),
                  map_location="cpu", weights_only=False)["value"]
BOUNDARY_HEADS = sorted(lh for lh, c in _aff.items() if c < BOUNDARY_COS_THR)
BOUNDARY_MASK  = mask_bin(BOUNDARY_HEADS)

UNION_HEADS = sorted(set(GOAL_HEADS) | set(BOUNDARY_HEADS))
print(f"GOAL_HEADS     : {len(GOAL_HEADS):3d}  (layers {min(l for l,_ in GOAL_HEADS)}–{max(l for l,_ in GOAL_HEADS)})")
print(f"BOUNDARY_HEADS : {len(BOUNDARY_HEADS):3d}  (layers {min(l for l,_ in BOUNDARY_HEADS)}–{max(l for l,_ in BOUNDARY_HEADS)})")
print(f"UNION_HEADS    : {len(UNION_HEADS):3d}")

GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [4]:
from src.tokens import prepare_input
from rich.style import Style

def exact_prompt(prompt, tail_tokens=None, patch_pos=[-1], emphasize_tokens=None):
    '''Prompt EXACTLY as the model sees it: PATCH_POS (pivot) in salmon, emphasize_tokens (TASK span) in yellow.'''
    ids = prepare_input(prompts=prompt, tokenizer=mt).input_ids[0].tolist()
    piv = {len(ids) + p if p < 0 else p for p in patch_pos}
    emph = set(emphasize_tokens or ())
    lo = 0 if tail_tokens is None else max(0, len(ids) - tail_tokens)
    OPEN, CLOSE, EOPEN, ECLOSE = "\x00<", "\x00>", "\x01<", "\x01>"
    parts = []
    for i in range(lo, len(ids)):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i in piv:      s = OPEN + s + CLOSE
        elif i in emph:   s = EOPEN + s + ECLOSE
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(OPEN, "[bold black on salmon1]").replace(CLOSE, "[/bold black on salmon1]")
              .replace(EOPEN, "[yellow]").replace(ECLOSE, "[/yellow]"))
    head = f"[grey42]… {lo} earlier tokens elided …[/grey42]\n" if lo else ""
    return head + txt

def pivot_token(prompt, patch_pos=[-1]):
    ids = prepare_input(prompts=prompt, tokenizer=mt).input_ids[0].tolist()
    return [mt.tokenizer.decode([ids[len(ids) + p if p < 0 else p]]) for p in patch_pos]

def token_span(prompt, substring):
    '''[lo, hi) token indices of `substring` inside the tokenization of `prompt` (via char offsets).'''
    enc = prepare_input(prompts=prompt, tokenizer=mt, return_offsets_mapping=True)
    offs = enc["offset_mapping"][0].tolist()
    ci = prompt.index(substring); cj = ci + len(substring)
    toks = [k for k, (a, b) in enumerate(offs) if b > a and a < cj and b > ci]
    assert toks, f"substring not found in tokenization: {substring!r}"
    return min(toks), max(toks) + 1

def show_prompt(tag, prompt, span):
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, patch_pos=[-1], emphasize_tokens=range(*span))),
                     title=f"[bold]{tag}[/bold]  TASK span {span}  ·  pivot={pivot_token(prompt)!r}  ·  "
                           f"L={prepare_input(prompts=prompt, tokenizer=mt)['input_ids'].shape[1]}",
                     title_align="left", border_style="grey42", padding=(1,1)))

## Shared primitives — prompt builders, unified q/k/v patch, readout helpers

These are the only pieces both sections share. Everything spec-specific (prompts, intervention, and the
three tests) lives in the SPEC-1 / SPEC-2 sections below, each with its own analysis.

In [5]:
import baukit
from src.attention import get_attention_matrices
from src.goals.goalscope import apply_goalscope

READOUT_HEADS = GOAL_HEADS                 # heads whose attention / OV write we read out (both sections)

# ── prompt builders ──
def get_prompt(question, prefill):
    messages = [
        {"role": "system", "content": "You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain."},
        {"role": "user", "content": question},
        {"role": "assistant", "content": prefill},
    ]
    return mt.tokenizer.apply_chat_template(messages, add_generation_prompt=False,
        continue_final_message=True, tokenize=False, chat_template_kwargs={"enable_thinking": False})

# ── q/k/v patch primitives ──
def _proj_name(l, which):                  # which in {'q','k','v'}
    return mt.attn_module_name_format.format(l) + f".{which}_proj"

def cache_src_proj(prompt, positions, layers, which):
    '''{layer: (len(positions), width)} — source {which}_proj outputs at `positions` (pre-RoPE for q/k).'''
    tok = prepare_input(prompts=prompt, tokenizer=mt)
    names = {l: _proj_name(l, which) for l in layers}
    with torch.no_grad(), baukit.TraceDict(mt._model, list(names.values()),
                                           retain_input=False, retain_output=True) as tr:
        mt._model(**{k: v for k, v in tok.items() if k in ("input_ids", "attention_mask")})
    return {l: tr[names[l]].output[0, positions].detach().float().cpu() for l in layers}

def heads_per_layer(head_set, which, layers=None):
    '''{layer: set(head)} — q-head index for which=="q", else kv-head index (h // GROUP).'''
    d = defaultdict(set)
    for (l, h) in head_set:
        if layers is None or l in layers:
            d[l].add(h if which == "q" else h // GROUP)
    return dict(d)

def _layers_of(head_set):
    hs = UNION_HEADS if head_set == "ALL" else head_set
    return sorted({l for l, _ in hs})

@contextmanager
def proj_patch_hooks(mt, patch_specs):
    '''Replace {which}_proj-output head-slices at `positions` with cached source vectors, for this forward.
    Positions absent from the current forward (e.g. a prompt position during a 1-token decode step) are
    skipped — so a prefill spec is a no-op on decode, and the effect then rides the KV cache.'''
    hd = mt.head_dim
    handles = []
    try:
        for spec in patch_specs:
            which, cache, headmap, positions = spec["which"], spec["cache"], spec["heads"], spec["positions"]
            for layer, hset in headmap.items():
                mod = baukit.get_module(mt._model, _proj_name(layer, which))
                src_l = cache[layer]           # (K, width)
                def make(hset, src_l, positions):
                    def hook(module, inp, out):
                        seq = out.shape[1]
                        pos = [p if p >= 0 else seq + p for p in positions]
                        keep = [(r, p) for r, p in enumerate(pos) if 0 <= p < seq]
                        if not keep:
                            return out
                        o = out.clone()
                        for h in hset:
                            sl = slice(h*hd, (h+1)*hd)
                            for r, p in keep:
                                o[:, p, sl] = src_l[r, sl].to(o.dtype).to(o.device)
                        return o
                    return hook
                handles.append(mod.register_forward_hook(make(hset, src_l, positions)))
        yield
    finally:
        for h in handles:
            h.remove()

def make_patch_interface(mt, patch_specs):
    def iface(mt, inputs, patches=None, model_kwargs=None):
        with proj_patch_hooks(mt, patch_specs):
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return iface

# ── readouts (generic; each section calls with its own args) ──
def pivot_attn(info, heads, positions, pivot):
    '''mean over `heads` of value-weighted attention from `pivot` onto `positions` -> (len(positions),).'''
    A = info.attention_matrices                # (L, H, q, s)
    return np.stack([A[l, h, pivot, positions] for (l, h) in heads]).mean(0)

def attn_table(title, toks, a_ref, a_patch, ref_name):
    tbl = Table(title=title, highlight=False)
    tbl.add_column("task token"); tbl.add_column(ref_name, justify="right")
    tbl.add_column("patched", justify="right"); tbl.add_column("Δ", justify="right")
    for t, c, p in zip(toks, a_ref, a_patch):
        d = p - c; col = "green" if d > 0 else "red"
        tbl.add_row(escape(repr(t)), f"{c:.3f}", f"{p:.3f}", f"[{col}]{d:+.3f}[/{col}]")
    rich_print(tbl)
    print(f"total mass on span:  {ref_name}={a_ref.sum():.3f}  patched={a_patch.sum():.3f}")

def goalscope_readout(tag, prompt, pivot, heads=None, iface=None,
                      strategy="additive_norm", alpha=0.1, layers=(35, 46), maxnew=128):
    res = apply_goalscope(prompt=prompt, mt=mt, selected_heads=heads or READOUT_HEADS,
        strategy=strategy, alpha=alpha, steer_layer_range=layers, batch_size=1, max_new_tokens=maxnew,
        goalscope_prompt=globals().get("gsp"), token_positions=[pivot],
        include_noncode=False, show=False, patch_interface=iface)
    for r in res:
        rich_print(f"[bold]{tag}[/bold]  pos={r['pos']} {r['token']!r} => [italic]{escape(r['patched_gen'])}[/italic]")

def greedy(tok, prefill_specs=None, decode_specs=None, n_steer=1, max_new=64):
    '''Greedy decode. `prefill_specs` patched on the prefill (rides the KV cache after); `decode_specs`
    re-applied at the current token for the first `n_steer` forward passes (prefill = pass 1).'''
    ids, am = tok["input_ids"], tok["attention_mask"]
    ctx0 = proj_patch_hooks(mt, prefill_specs) if prefill_specs else nullcontext()
    with torch.no_grad(), ctx0:
        out = mt._model(input_ids=ids, attention_mask=am, use_cache=True)
    pkv = out.past_key_values
    nxt = out.logits[:, -1].argmax(-1, keepdim=True); gen = [int(nxt)]
    with torch.no_grad():
        for step in range(1, max_new):
            am = torch.cat([am, torch.ones_like(nxt)], 1)
            use = decode_specs if (decode_specs and step < n_steer) else None
            ctx = proj_patch_hooks(mt, use) if use else nullcontext()
            with ctx:
                out = mt._model(input_ids=nxt, attention_mask=am, past_key_values=pkv, use_cache=True)
            pkv = out.past_key_values
            nxt = out.logits[:, -1].argmax(-1, keepdim=True); t = int(nxt)
            if t == mt.tokenizer.eos_token_id:
                break
            gen.append(t)
    return mt.tokenizer.decode(gen, skip_special_tokens=True)

# ── delta-attention viz (red = negative Δ, green = positive Δ; symmetric about 0) ──
from src.attention import _safe_token
from circuitsvis.tokens import colored_tokens
from IPython.display import display

def show_delta_tokens(tokens, delta, label=""):
    '''Diverging red(neg)/green(pos) per-token coloring of an attention delta, symmetric about 0.'''
    d = np.asarray(delta, dtype=float)
    M = float(np.abs(d).max()) or 1.0
    if label:
        rich_print(f"[grey58]{label}   (green +Δ / red −Δ, |Δ|max={M:.3f})[/grey58]")
    display(colored_tokens(tokens=[_safe_token(t) for t in tokens], values=d,
                           min_value=-M, max_value=M, negative_color="red", positive_color="green"))

---
# SPEC 1 — same TASK, different PROGRESS  ·  QUERY transplant @ pivot

Shared TASK (*longest word is a palindrome*); SRC is further along (`…[-1]` already selected) than DEST.
Only the pivot **query** differs → transplant SRC's pivot query into DEST. Keys/values (task) are shared,
identical in tokens *and positions*, so they are untouched. **Prediction: patched DEST ≈ clean SRC.**

In [6]:
QUESTION_TMPL = '''Write a Python program that implements a function `func` which, given a list of strings `words`, computes:
<<task>>
'''
PREFILL_TMPL  = '''<code>
def func(words: list[str]):
    res = {progress}
    return [word for word in res if'''
# PREFILL_TMPL  = '''<code>
# def func(words: list[str]):
#     res = {progress}
#     return'''

# ── SPEC-1 prompts ──
TASK_1          = "return all the words in `words` that are palindromes and starts with a vowel"
SRC_PROGRESS_1  = "[word for word in words if word == word[::-1]]"   
DEST_PROGRESS_1 = "[word for word in words if word.lower()[0] in 'aeiou']"
SRC_PROGRESS_1, DEST_PROGRESS_1 = DEST_PROGRESS_1, SRC_PROGRESS_1

src_prompt_1  = get_prompt(QUESTION_TMPL.replace("<<task>>", TASK_1), PREFILL_TMPL.format(progress=SRC_PROGRESS_1))
dest_prompt_1 = get_prompt(QUESTION_TMPL.replace("<<task>>", TASK_1), PREFILL_TMPL.format(progress=DEST_PROGRESS_1))
SRC_TASK_SPAN_1  = token_span(src_prompt_1,  TASK_1)
DEST_TASK_SPAN_1 = token_span(dest_prompt_1, TASK_1)   # identical positions to SRC (task precedes the prefill)
L_src_1  = prepare_input(prompts=src_prompt_1,  tokenizer=mt)["input_ids"].shape[1]
L_dest_1 = prepare_input(prompts=dest_prompt_1, tokenizer=mt)["input_ids"].shape[1]
PIVOT_SRC_1, PIVOT_DEST_1 = L_src_1 - 1, L_dest_1 - 1

show_prompt("SOURCE", src_prompt_1,  SRC_TASK_SPAN_1)
show_prompt("DEST",   dest_prompt_1, DEST_TASK_SPAN_1)
print(f"pivot query: src[{PIVOT_SRC_1}]={pivot_token(src_prompt_1)!r} -> dest[{PIVOT_DEST_1}]={pivot_token(dest_prompt_1)!r}")
print(f"task spans (should match): SRC={SRC_TASK_SPAN_1}  DEST={DEST_TASK_SPAN_1}")

╭─ SOURCE  TASK span (63, 82)  ·  pivot=[' if']  ·  L=127 ────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ return all the words in `words` that are palindromes and starts with a vowel\n                                  │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [word for word in words if word.lower()[0] in 'aeiou']\n                                              │
│     return [word for word in res if                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DEST  TASK span (63, 82)  ·  pivot=[' if']  ·  L=123 ──────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ return all the words in `words` that are palindromes and starts with a vowel\n                                  │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     res = [word for word in words if word == word[::-1]]\n                                                      │
│     return [word for word in res if                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

pivot query: src[126]=[' if'] -> dest[122]=[' if']
task spans (should match): SRC=(63, 82)  DEST=(63, 82)


In [7]:
# ── SPEC-1 intervention: QUERY @ pivot ──
PATCH_HEADS_1 = UNION_HEADS                 # GOAL_HEADS | BOUNDARY_HEADS | UNION_HEADS | "ALL"
PATCH_LAYERS_1 = _layers_of(PATCH_HEADS_1)
READOUT_HEADS_1 = PATCH_HEADS_1   # read out / delta / goalscope over the SAME heads we patch

qheads_1 = heads_per_layer(PATCH_HEADS_1, "q", layers=set(PATCH_LAYERS_1))
q_cache_1 = cache_src_proj(src_prompt_1, [PIVOT_SRC_1], sorted(qheads_1), "q")

prefill_specs_1 = [dict(which="q", cache=q_cache_1, heads=qheads_1, positions=[PIVOT_DEST_1])]
decode_specs_1  = [dict(which="q", cache=q_cache_1, heads=qheads_1, positions=[-1])]   # re-inject at each new token
patch_iface_1   = make_patch_interface(mt, prefill_specs_1)                            # single-forward readouts

_nh = sum(len(v) for v in qheads_1.values())
print(f"SPEC 1: patch QUERY · {_nh} (layer,q-head) slices across {len(PATCH_LAYERS_1)} layers "
      f"[{PATCH_LAYERS_1[0]}–{PATCH_LAYERS_1[-1]}] · heads={PATCH_HEADS_1 if isinstance(PATCH_HEADS_1,str) else len(PATCH_HEADS_1)}")

SPEC 1: patch QUERY · 228 (layer,q-head) slices across 43 layers [11–62] · heads=228


## 1A · Test A — value-weighted attention (patched vs **clean DEST**, over the shared TASK)

In [8]:
dest_tok_1 = prepare_input(prompts=dest_prompt_1, tokenizer=mt)
_ids1 = dest_tok_1["input_ids"][0]
pos_1 = list(range(*DEST_TASK_SPAN_1))

info_clean_1 = get_attention_matrices(dest_tok_1, mt, value_weighted=True)
info_patch_1 = get_attention_matrices(dest_tok_1, mt, value_weighted=True, patch_interface=patch_iface_1)
a_clean_1 = pivot_attn(info_clean_1, READOUT_HEADS_1, pos_1, PIVOT_DEST_1)
a_patch_1 = pivot_attn(info_patch_1, READOUT_HEADS_1, pos_1, PIVOT_DEST_1)
toks_1 = [mt.tokenizer.decode([_ids1[j]]) for j in pos_1]
attn_table(f"SPEC 1 · pivot→TASK  (mean over {len(READOUT_HEADS_1)} readout heads)", toks_1, a_clean_1, a_patch_1, "clean DEST")
show_delta_tokens(toks_1, a_patch_1 - a_clean_1, "SPEC 1 · Δattn pivot→TASK (patched − clean DEST)")

 SPEC 1 · pivot→TASK  (mean over 228 readout  
                    heads)                    
┏━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓
┃ task token ┃ clean DEST ┃ patched ┃      Δ ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━┩
│ 'return'   │      0.005 │   0.007 │ +0.002 │
│ ' all'     │      0.014 │   0.020 │ +0.006 │
│ ' the'     │      0.008 │   0.013 │ +0.006 │
│ ' words'   │      0.021 │   0.050 │ +0.030 │
│ ' in'      │      0.008 │   0.009 │ +0.001 │
│ ' `'       │      0.004 │   0.003 │ -0.002 │
│ 'words'    │      0.007 │   0.013 │ +0.005 │
│ '`'        │      0.005 │   0.007 │ +0.002 │
│ ' that'    │      0.026 │   0.036 │ +0.010 │
│ ' are'     │      0.014 │   0.028 │ +0.015 │
│ ' pal'     │      0.060 │   0.198 │ +0.138 │
│ 'ind'      │      0.084 │   0.765 │ +0.681 │
│ 'rom'      │      0.052 │   0.160 │ +0.109 │
│ 'es'       │      0.072 │   0.139 │ +0.066 │
│ ' and'     │      0.053 │   0.069 │ +0.015 │
│ ' starts'  │      0.132 │   0.070 │ -0.062 │
│ ' with'    │      0.184 │   0.067 │ -0.117 │
│ ' a'       │      0.098 │   0.020 │ -0.079 │
│ ' vowel'   │      1.494 │   0.245 │ -1.249 │
└────────────┴────────────┴─────────┴────────┘

total mass on span:  clean DEST=2.341  patched=1.918


SPEC 1 · Δattn pivot→TASK (patched − clean DEST)   (green +Δ / red −Δ, |Δ|max=1.249)

## 1B · Test B — goalscope readout  (prediction: patched DEST ≈ clean SRC)

In [9]:
goalscope_readout("clean SRC    ", src_prompt_1,  PIVOT_SRC_1,  heads=READOUT_HEADS_1)
goalscope_readout("clean DEST   ", dest_prompt_1, PIVOT_DEST_1, heads=READOUT_HEADS_1)
goalscope_readout("patched DEST ", dest_prompt_1, PIVOT_DEST_1, heads=READOUT_HEADS_1, iface=patch_iface_1)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


clean SRC      pos=126 ' if' => I'm checking if the string is equal to its reverse (palindrome check).

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

clean DEST     pos=122 ' if' => I'm checking if the character is a vowel (a, e, i, o, u) and also if it's 
uppercase.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

patched DEST   pos=122 ' if' => I'm checking if the string is equal to its reverse (palindrome check).

## 1C · Test C — generation, `N_STEER` query re-injection

The query is not cached, so persistence needs re-injection: the SAME `PIVOT_SRC` query is patched at the
current token for the first `N_STEER` forward passes (the "weird but for-now" scheme). `N_STEER=1` patches
only the prefill pivot.

In [10]:
N_STEER_1 = 50

src_tok_1 = prepare_input(prompts=src_prompt_1, tokenizer=mt)
rich_print(f"[grey58]N_STEER={N_STEER_1}[/grey58]")
rich_print(f"[sea_green3]clean SRC     :[/sea_green3] {escape(repr(greedy(src_tok_1)))}")
rich_print(f"[grey70]clean DEST    :[/grey70] {escape(repr(greedy(dest_tok_1)))}")
rich_print(f"[salmon1]patched DEST  :[/salmon1] "
           f"{escape(repr(greedy(dest_tok_1, prefill_specs=prefill_specs_1, decode_specs=decode_specs_1, n_steer=N_STEER_1)))}")

N_STEER=50

clean SRC     : ' word.lower() == word[::-1].lower()]\n\n</code>'

clean DEST    : " word[0].lower() in 'aeiou']\n</code>"

patched DEST  : ' word == word[::-1] and word.lower() == word.lower()[::-1]]\n</code>'

---
# SPEC 2 — different TASK, same PROGRESS  ·  KEY + VALUE transplant @ task span

Shared PROGRESS (`sorted(words, key=len)`); tasks differ. Transplant SRC's task **keys + values** into
DEST's task span (right-edge aligned, K clipped). KEY re-addresses, VALUE re-fills the content.
**Prediction: patched DEST ≈ clean SRC.**

In [11]:
QUESTION_TMPL = '''Write a Python program that implements a function `func` which, given a list of strings `words`, computes:
<<task>>
'''
# PREFILL_TMPL  = '''<code>
# def func(words: list[str]):
#     res = {progress}
#     return'''
PREFILL_TMPL  = '''<code>
def func(words: list[str]):
    {progress}
    '''
# PREFILL_TMPL  = '''<code>
# def func(nums: list[int]):
#     res = {progress}
#     '''

QUESTION_TMPL_CPP = '''Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:
<<task>>
Assume that all the libraries have been already imported.
'''
PREFILL_TMPL_CPP  = '''<code>
auto func(std::vector<std::string> words){
    <<progress>>
    '''

# ── SPEC-2 prompts ──
# PROGRESS_2_SRC  = "sorted(words, key=len)"
# PROGRESS_2_DEST = PROGRESS_2_SRC
# SRC_TASK_2  = "returns true if the shortest word has all unique letters, false otherwise"
# DEST_TASK_2 = "returns true if the longest word is a palindrome, false otherwise"
# ---
# SRC_TASK_2 = "find the longest word in words that is a palindrome"
# PROGRESS_2_SRC = "[w for w in words if w == w[::-1]]"
# DEST_TASK_2 = "find the shortest word in words that has all unique letters"
# PROGRESS_2_DEST = "[w for w in words if len(set(w)) == len(w)]"
# ---
#! SAME TASK. Shouldn't change things
# SRC_TASK_2 = "return all the words in `words` that are palindromes and starts with a vowel"
# DEST_TASK_2 = SRC_TASK_2
# PROGRESS_2_SRC = "[w for w in words if w == w[::-1]]"
# PROGRESS_2_DEST = "[w for w in words if w[0].lower() in 'aeiou']"
# ---
# SRC_TASK_2  = "returns true if the shortest word has all unique letters, false otherwise"
# SRC_TASK_2 = "returns true if the shortest word starts with a vowel."

# SRC_TASK_2 = "returns true if all the words have the same length, false otherwise"
# PROGRESS_2_SRC  = "sorted(words, key=len)"
# DEST_TASK_2 = "returns true if the longest word is a palindrome, false otherwise"
# PROGRESS_2_DEST = """std::stable_sort(words.begin(), words.end(), [](const std::string& a, const std::string& b) {
#         return a.size() < b.size();
#     });"""
# ---
# SRC_TASK_2 = "the number of words that have at least two characters"
# PROGRESS_2_SRC  = "long_words = [w for w in words if len(w) >= 2]"
# DEST_TASK_2 = "the number of words with two characters that start and end with the same letter"
# PROGRESS_2_DEST = """std::vector<std::string> long_words;
#     for (const auto& w : words) if (w.size() >= 2) long_words.push_back(w);"""
#--
# SRC_TASK_2 = """(1) the integer mean of the values
# (2) the number of odd values
# (3) the number of even values
# """
# PROGRESS_2_SRC  = """a = sum(nums) // len(nums)
#     b = sum(1 for x in nums if x % 2 != 0)"""
# DST_TASK_2 = """(1) the integer mean of the values
# (2) the number of odd values
# (3) the product of all values
# """
# PROGRESS_2_DEST = """a = sum(nums) // len(nums)
#     b = sum(1 for x in nums if x % 2 != 0)"""
# ---
SRC_TASK_2 = """(1) the number of capitalized words
(2) the length of the longest word"""
PROGRESS_2_SRC = """a = sum(1 for w in words if w[0].isupper())"""
DEST_TASK_2 = """(1) the number of capitalized words
(2) whether any word is empty"""
PROGRESS_2_DEST = PROGRESS_2_SRC

# ======
# SRC_TASK_2, DEST_TASK_2 = DEST_TASK_2, SRC_TASK_2
# PROGRESS_2_SRC, PROGRESS_2_DEST = PROGRESS_2_DEST, PROGRESS_2_SRC

src_prompt_2  = get_prompt(QUESTION_TMPL.replace("<<task>>", SRC_TASK_2),  PREFILL_TMPL.format(progress=PROGRESS_2_SRC))
dest_prompt_2 = get_prompt(QUESTION_TMPL.replace("<<task>>", DEST_TASK_2), PREFILL_TMPL.format(progress=PROGRESS_2_DEST))
# dest_prompt_2 = get_prompt(QUESTION_TMPL_CPP.replace("<<task>>", DEST_TASK_2), PREFILL_TMPL_CPP.replace("<<progress>>", PROGRESS_2_DEST))

SRC_TASK_SPAN_2  = token_span(src_prompt_2,  SRC_TASK_2)
DEST_TASK_SPAN_2 = token_span(dest_prompt_2, DEST_TASK_2)
L_src_2  = prepare_input(prompts=src_prompt_2,  tokenizer=mt)["input_ids"].shape[1]
L_dest_2 = prepare_input(prompts=dest_prompt_2, tokenizer=mt)["input_ids"].shape[1]
PIVOT_SRC_2, PIVOT_DEST_2 = L_src_2 - 1, L_dest_2 - 1

K2 = min(SRC_TASK_SPAN_2[1]-SRC_TASK_SPAN_2[0], DEST_TASK_SPAN_2[1]-DEST_TASK_SPAN_2[0])
SRC_POS_2  = list(range(SRC_TASK_SPAN_2[1]-K2,  SRC_TASK_SPAN_2[1]))   # source key/value rows to cache
DEST_POS_2 = list(range(DEST_TASK_SPAN_2[1]-K2, DEST_TASK_SPAN_2[1]))  # dest positions to overwrite

show_prompt("SOURCE", src_prompt_2,  SRC_TASK_SPAN_2)
show_prompt("DEST",   dest_prompt_2, DEST_TASK_SPAN_2)
print(f"SRC  task→query distance: {PIVOT_SRC_2 - SRC_TASK_SPAN_2[1]} .. {PIVOT_SRC_2 - SRC_TASK_SPAN_2[0]}")
print(f"DEST task→query distance: {PIVOT_DEST_2 - DEST_TASK_SPAN_2[1]} .. {PIVOT_DEST_2 - DEST_TASK_SPAN_2[0]}")
print(f"right-edge aligned: K={K2} key/value tokens  src{SRC_POS_2} -> dest{DEST_POS_2}")

╭─ SOURCE  TASK span (63, 81)  ·  pivot=['    ']  ·  L=116 ───────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of capitalized words\n                                                                           │
│ (2) the length of the longest word\n                                                                            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w[0].isupper())\n                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DEST  TASK span (63, 80)  ·  pivot=['    ']  ·  L=115 ─────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of capitalized words\n                                                                           │
│ (2) whether any word is empty\n                                                                                 │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w[0].isupper())\n                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SRC  task→query distance: 34 .. 52
DEST task→query distance: 34 .. 51
right-edge aligned: K=17 key/value tokens  src[64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80] -> dest[63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]


In [12]:
# ── SPEC-2 intervention: KEY + VALUE @ task span ──
PATCH_HEADS_2 = GOAL_HEADS                 # GOAL_HEADS | BOUNDARY_HEADS | UNION_HEADS | "ALL"
PATCH_LAYERS_2 = _layers_of(PATCH_HEADS_2)
READOUT_HEADS_2 = PATCH_HEADS_2   # read out / delta / goalscope over the SAME heads we patch

kvheads_2 = heads_per_layer(PATCH_HEADS_2, "k", layers=set(PATCH_LAYERS_2))
k_cache_2 = cache_src_proj(src_prompt_2, SRC_POS_2, sorted(kvheads_2), "k")
v_cache_2 = cache_src_proj(src_prompt_2, SRC_POS_2, sorted(kvheads_2), "v")

patch_specs_2 = [dict(which="k", cache=k_cache_2, heads=kvheads_2, positions=DEST_POS_2),
                 dict(which="v", cache=v_cache_2, heads=kvheads_2, positions=DEST_POS_2)]
patch_iface_2 = make_patch_interface(mt, patch_specs_2)

_nh = sum(len(v) for v in kvheads_2.values())
print(f"SPEC 2: patch KEY+VALUE · {_nh} (layer,kv-head) slices across {len(PATCH_LAYERS_2)} layers "
      f"[{PATCH_LAYERS_2[0]}–{PATCH_LAYERS_2[-1]}] · heads={PATCH_HEADS_2 if isinstance(PATCH_HEADS_2,str) else len(PATCH_HEADS_2)}")
if GQA_BLEED:
    print(f"NOTE (GQA bleed): each patched kv-head is shared by {GROUP} query heads.")

SPEC 2: patch KEY+VALUE · 70 (layer,kv-head) slices across 26 layers [28–55] · heads=122


## 2B · Test B — goalscope readout  (prediction: patched DEST ≈ clean SRC = SRC.DIFF)

The goal-head write is extracted at the DEST pivot **under the intervention**, then verbalized. With `GQA_BLEED=False` the extraction runs under **headwise routing** (only `PATCH_HEADS_2` read the source span — same intervention as 2C), so the readout is not contaminated by the bled non-goal heads propagating through the residual. `GQA_BLEED=True` uses the raw per-kv-head patch (bled).

In [13]:
from src.diff.functional import make_headwise_interface   # new name; import * doesn't re-bind on autoreload
_g_iface = (patch_iface_2 if GQA_BLEED
            else make_headwise_interface(mt, PATCH_HEADS_2, k_cache_2, v_cache_2, DEST_POS_2))
rich_print(f"[grey58]patched-DEST readout extracted under "
           f"{'raw k/v patch (GQA bleed)' if GQA_BLEED else 'headwise routing (GQA-corrected)'}[/grey58]")
goalscope_readout("clean SRC    ", src_prompt_2,  PIVOT_SRC_2,  heads=READOUT_HEADS_2)
goalscope_readout("clean DEST   ", dest_prompt_2, PIVOT_DEST_2, heads=READOUT_HEADS_2)
goalscope_readout("patched DEST ", dest_prompt_2, PIVOT_DEST_2, heads=READOUT_HEADS_2, iface=_g_iface)

patched-DEST readout extracted under headwise routing (GQA-corrected)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

clean SRC      pos=115 '    ' => I'm calculating the length of each string in the list and finding the maximum 
value.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

clean DEST     pos=114 '    ' => I'm checking if the current line is empty or contains only whitespace.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

patched DEST   pos=114 '    ' => I'm checking if the current line is empty or contains only whitespace characters.

## 2C · Test C — generation

**`GQA_BLEED=True` (raw patch):** the task-span keys/values are written into the KV-cache at prefill and ride it for the whole generation — but they bleed to **all `GROUP` query heads** sharing each patched kv-head, so the effect is not confined to `PATCH_HEADS_2`.

**`GQA_BLEED=False` (default, corrected):** per-query-head routing — at each patched layer only the selected query heads read the source span; all other heads read the clean dest k/v. One trajectory with the correct propagated query at every layer, so it fixes both token-divergence and query-propagation. Per-query-head routing can't ride the per-kv-head cache, so each step runs **two attention passes** (clean + source-span) at the patched layers; the cache itself stays clean.

In [14]:
from src.diff.functional import headwise_kv_patch   # autoreload doesn't re-bind new star-imported names

dest_tok_2 = prepare_input(dest_prompt_2, tokenizer=mt)
src_tok_2 = prepare_input(src_prompt_2, tokenizer=mt)
if GQA_BLEED:
    rich_print("[grey58]GQA_BLEED=True: raw per-kv-head patch — bleeds to all GROUP query heads; K+V ride the KV-cache[/grey58]")
    _gen = greedy(dest_tok_2, prefill_specs=patch_specs_2)
else:
    rich_print("[grey58]GQA_BLEED=False: per-query-head routing — only PATCH_HEADS_2 read source span (2 attn passes/step, cache stays clean)[/grey58]")
    with headwise_kv_patch(mt, PATCH_HEADS_2, k_cache_2, v_cache_2, DEST_POS_2):
        _gen = greedy(dest_tok_2)

rich_print(f"[sea_green3]clean SRC     :[/sea_green3] {escape(repr(greedy(src_tok_2)))}")
rich_print(f"[grey70]clean DEST    :[/grey70] {escape(repr(greedy(dest_tok_2)))}")
rich_print(f"[salmon1]patched DEST  :[/salmon1] {escape(repr(_gen))}")

GQA_BLEED=False: per-query-head routing — only PATCH_HEADS_2 read source span (2 attn passes/step, cache stays 
clean)

clean SRC     : ' b = max(len(w) for w in words)\n    return a, b\n</code>'

clean DEST    : ' b = any(not w for w in words)\n    return a, b\n</code>'

patched DEST  : ' b = any(len(w) == 0 for w in words)\n    return a, b\n</code>'

### 2C-validate · headwise routing sanity check
`headwise_kv_patch(route_group=True)` routes **every** query head of each patched group to the source span — that is exactly the raw bled patch. So its generation must match `greedy(prefill_specs=patch_specs_2)` token-for-token. If it does, the routing machinery (RoPE of the source span, per-head splice) is correct, and the `route_group=False` result is the genuine GQA-bleed-corrected generation.

In [15]:
from src.diff.functional import headwise_kv_patch   # autoreload doesn't re-bind new star-imported names
# equivalence check: route_group=True should reproduce the raw per-kv-head patch
_raw = greedy(dest_tok_2, prefill_specs=patch_specs_2)
with headwise_kv_patch(mt, PATCH_HEADS_2, k_cache_2, v_cache_2, DEST_POS_2, route_group=True):
    _grp = greedy(dest_tok_2)
rich_print(f"[grey70]raw bled patch      :[/grey70] {escape(repr(_raw))}")
rich_print(f"[grey70]headwise route_group:[/grey70] {escape(repr(_grp))}")
rich_print(f"[{'sea_green3' if _raw == _grp else 'red'}]match: {_raw == _grp}[/]  "
           f"(True ⇒ routing machinery is correct)")

raw bled patch      : ' b = any(len(w) == 0 for w in words)\n    return a, b\n</code>'

headwise route_group: ' b = any(len(w) == 0 for w in words)\n    return a, b\n</code>'

match: True  (True ⇒ routing machinery is correct)